"""
we’ll switch to another interesting data set: data from a Washington,
D.C., bike sharing system reporting the hourly count of rental bikes between
2011 and 2012 in the Capital bike-share system with the corresponding weather and
seasonal information.
The goal is to take a flat 2D data set and transform it into a 3D one, as shown in
figure 3.2.
In the source data, each row is a separate hour of data (Figure 3.2 shows a transposed
version of this to better fit on the printed page).
Figure 3.2 Transforming a 1D multichannel data set into a 2D multichannel data set by separating the date
and hour of each sample into separate axes
We want to change the row-per-hour organization so that you have one axis that increases at a rate of one day per index
increment and another axis that represents hour of day (independent of the date). The
third axis is different columns of data (weather, temperature, and so on).
Load the data, as shown in the following listing.
"""

In [4]:
import numpy as np
import torch
import csv

bikes_numpy = np.loadtxt("hour-fixed.csv",
                        dtype = np.float32,
                        delimiter=",",
                        skiprows=1,
                        converters={1: lambda x: float(x[8:10])})
bikes_numpy.shape, bikes_numpy

((17520, 17),
 array([[1.0000e+00, 1.0000e+00, 1.0000e+00, ..., 3.0000e+00, 1.3000e+01,
         1.6000e+01],
        [2.0000e+00, 1.0000e+00, 1.0000e+00, ..., 8.0000e+00, 3.2000e+01,
         4.0000e+01],
        [3.0000e+00, 1.0000e+00, 1.0000e+00, ..., 5.0000e+00, 2.7000e+01,
         3.2000e+01],
        ...,
        [1.7377e+04, 3.1000e+01, 1.0000e+00, ..., 7.0000e+00, 8.3000e+01,
         9.0000e+01],
        [1.7378e+04, 3.1000e+01, 1.0000e+00, ..., 1.3000e+01, 4.8000e+01,
         6.1000e+01],
        [1.7379e+04, 3.1000e+01, 1.0000e+00, ..., 1.2000e+01, 3.7000e+01,
         4.9000e+01]], dtype=float32))

In [5]:
bikes = torch.from_numpy(bikes_numpy)
bikes.shape, bikes

(torch.Size([17520, 17]),
 tensor([[1.0000e+00, 1.0000e+00, 1.0000e+00,  ..., 3.0000e+00, 1.3000e+01,
          1.6000e+01],
         [2.0000e+00, 1.0000e+00, 1.0000e+00,  ..., 8.0000e+00, 3.2000e+01,
          4.0000e+01],
         [3.0000e+00, 1.0000e+00, 1.0000e+00,  ..., 5.0000e+00, 2.7000e+01,
          3.2000e+01],
         ...,
         [1.7377e+04, 3.1000e+01, 1.0000e+00,  ..., 7.0000e+00, 8.3000e+01,
          9.0000e+01],
         [1.7378e+04, 3.1000e+01, 1.0000e+00,  ..., 1.3000e+01, 4.8000e+01,
          6.1000e+01],
         [1.7379e+04, 3.1000e+01, 1.0000e+00,  ..., 1.2000e+01, 3.7000e+01,
          4.9000e+01]]))

"""
For every hour, the data set reports the following variables:
instant           # index of record
day               # day of month
season            # season (1: spring, 2: summer, 3: fall, 4: winter)
yr                # year (0: 2011, 1: 2012)
mnth              # month (1 to 12)
hr                # hour (0 to 23)
holiday           # holiday status
weekday           # day of the week
workingday        # working day status
weathersit        # weather situation
                  # (1: clear, 2:mist, 3: light rain/snow, 4: heavy rain/snow)
temp              # temperature in C
atemp             # perceived temperature in C
hum               # humidity
windspeed         # windspeed
casual            # number of causal users
registered        # number of registered users
cnt               # count of rental bikes

"""

"""
In a time-series data set such as this one, rows represent successive time points: a
dimension along which they’re ordered. Sure, you could treat each row as independent
and try to predict the number of circulating bikes based on, say, a particular time
of day regardless of what happened earlier.
This existence of an ordering, however, gives you the opportunity to exploit causal
relationships across time. You can predict bike rides at one time based on the fact that
it was raining at an earlier time, for example. For the time being, you’re going to focus
on learning how to turn your bike-sharing data set into something that your neural
network can ingest in fixed-size chunks.
This neural network model needs to see sequences of values for each quantity,
such as ride count, time of day, temperature, and weather conditions, so N parallel
sequences of size C. C stands for channel, in neural network parlance, and is the same
as column for 1D data like you have here. The N dimension represents the time axis—
here, one entry per hour.
You may want to break up the 2-year data set in wider observation periods, such as
days. This way, you’ll have N (for number of samples) collections of C sequences of length
L. In other words, your time-series data set is a tensor of dimension 3 and shape N x C
x L. The C remains your 17 channels, and L would be 24, one per hour of the day.
There’s no particular reason why we must use chunks of 24 hours, though the general
daily rhythm is likely to give us patterns we can exploit for predictions. We could
instead use 7*24=168 hour blocks to chunk by week instead, if we desired.

Now go back to your bike-sharing data set. The first column is the index (the
global ordering of the data); the second is the date; the sixth is the time of day. You
have everything you need to create a data set of daily sequences of ride counts and
other exogenous variables. Your data set is already sorted, but if it weren’t, you could
use torch.sort on it to order it appropriately.

NOTE The version of the file you’re using here, hour-fixed.csv, has had
some processing done to include rows that were missing from the original
data set. We presumed that the missing hours had zero bikes active (typically
the early-morning hours).
All you have to do to obtain your daily hours data set is view the same tensor in
batches of 24 hours. Take a look at the shape and strides of your bikes tensor:
"""

In [2]:
bikes.shape, bikes.stride()

(torch.Size([17520, 17]), (17, 1))

"""
Tensor.stride(dim) → tuple or int
Returns the stride of self tensor.

Stride is the jump necessary to go from one element to the next one in the specified dimension dim. A tuple of all strides is returned when no argument is passed in. Otherwise, an integer value is returned as the stride in the particular dimension dim.

Parameters:
dim (int, optional) – the desired dimension in which stride is required

Example:

>>> x = torch.tensor([[1, 2, 3, 4, 5], [6, 7, 8, 9, 10]])
>>> x.stride()
(5, 1)
>>> x.stride(0)
5
>>> x.stride(-1)
1



"""

In [5]:
"""
That’s 17,520 hours, 17 columns. Now reshape the data to have three axes (day, hour,
and then your 17 columns):
"""
daily_bikes = bikes.view(-1, 24, bikes.shape[1])
daily_bikes.shape, daily_bikes.stride()

(torch.Size([730, 24, 17]), (408, 17, 1))

In [ ]:
"""
17 x 24 =408

17520 / 24 = 730

"""

In [ ]:
"""
What happened here? First, the bikes.shape[1] is 17, which is the number of columns
in the bikes tensor. But the real crux of the code is the call to view, which is important:
it changes the way that the tensor looks at the same data as contained in storage.
Calling view on a tensor returns a new tensor that changes the number of dimensions
and the striding information without changing the storage. As a result, you can
rearrange your tensor at zero cost because no data is copied at all. Your call to view
requires you to provide the new shape for the returned tensor. Use the -1 as a placeholder
for “however many indexes are left, given the other dimensions and the original
number of elements.”

Remember that Storage is a contiguous, linear container for numbers—floatingpoint,
in this case. Your bikes tensor has rows stored one after the other in corresponding
storage, as confirmed by the output from the call to bikes.stride() earlier.

Remember that Storage is a contiguous, linear container for numbers—floatingpoint,
in this case. Your bikes tensor has rows stored one after the other in corresponding
storage, as confirmed by the output from the call to bikes.stride() earlier.
For daily_bikes, stride is telling you that advancing by 1 along the hour dimension
(the second) requires you to advance by 17 places in the storage (or one set of
columns), whereas advancing along the day dimension (the first) requires you to
advance by a number of elements equal to the length of a row in the storage times 24
(here, 408, which is 17 * 24).

The rightmost dimension is the number of columns in the original data set. In the
middle dimension, you have time split into chunks of 24 sequential hours. In other
words, you now have N sequences of L hours in a day for C channels. To get to your
desired NxCxL ordering, you need to transpose the tensor:

"""

In [6]:
daily_bikes = daily_bikes.transpose(1, 2)
daily_bikes.shape, daily_bikes.stride()

(torch.Size([730, 17, 24]), (408, 1, 17))

In [ ]:
"""
We mentioned earlier that the weather-situation variable is ordinal. In fact, it has 4 levels:
1 for the best weather and 4 for the worst. You could treat this variable as categorical,
with levels interpreted as labels, or continuous. If you choose categorical, you turn
the variable into a one-hot encoded vector and concatenate the columns with the data
set. To make rendering your data easier, limit yourself to the first day for now. First,
initialize a zero-filled matrix with a number of rows equal to the number of hours in
the day and a number of columns equal to the number of weather levels:

"""

In [7]:
first_day = bikes[:24].long()
weather_onehot = torch.zeros(first_day.shape[0], 4)
first_day[:,9]

tensor([1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 2, 2, 2, 2])

In [ ]:
"""
column - 9 of the data:
weathersit # weather situation
# (1: clear, 2:mist, 3: light rain/snow, 4: heavy rain/snow)
"""

In [8]:
first_day.shape

torch.Size([24, 17])

In [88]:
# weather_onehot.shape        # torch.Size([24, 4])
# weather_onehot

In [ ]:
"""
Then scatter ones into our matrix according to the corresponding level at each row.
Remember the use of unsqueeze to add a singleton dimension earlier:
"""

In [12]:
weather_onehot.scatter_(dim=1, index=first_day[:,9].unsqueeze(1) - 1,value=1.0)
                        
# You’re decreasing the values by 1 because the weather situation ranges from 1 to 4,
# whereas indices are 0-based.

tensor([[1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 1., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 1., 0., 0.]])

In [15]:
"""
The day started with weather 1 and ended with 2, so that seems right.
Last, concatenate your matrix to your original data set, using the cat function.
Look at the first of your results
"""

'\nThe day started with weather 1 and ended with 2, so that seems right.\nLast, concatenate your matrix to your original data set, using the cat function.\nLook at the first of your results\n'

In [16]:
torch.cat((bikes[:24], weather_onehot), 1)[:1]

tensor([[ 1.0000,  1.0000,  1.0000,  0.0000,  1.0000,  0.0000,  0.0000,  6.0000,
          0.0000,  1.0000,  0.2400,  0.2879,  0.8100,  0.0000,  3.0000, 13.0000,
         16.0000,  1.0000,  0.0000,  0.0000,  0.0000]])

In [17]:
"""
Here, you prescribed your original bikes data set and your one-hot encoded weathersituation
matrix to be concatenated along the column dimension (such as 1). In other
words, the columns of the two data sets are stacked together, or the new one-hot
encoded columns are appended to the original data set. For cat to succeed, the tensors
must have the same size along the other dimensions (the row dimension, in this
case).
Note that your new last four columns are 1, 0, 0, 0—exactly what you’d expect
with a weather value of 1.

You could have done the same thing with the reshaped daily_bikes tensor.
Remember that it’s shaped (B, C, L), where L = 24. First, create the zero tensor, with
the same B and L but with the number of additional columns as C:
"""

'\nHere, you prescribed your original bikes data set and your one-hot encoded weathersituation\nmatrix to be concatenated along the column dimension (such as 1). In other\nwords, the columns of the two data sets are stacked together, or the new one-hot\nencoded columns are appended to the original data set. For cat to succeed, the tensors\nmust have the same size along the other dimensions (the row dimension, in this\ncase).\nNote that your new last four columns are 1, 0, 0, 0—exactly what you’d expect\nwith a weather value of 1.\n\nYou could have done the same thing with the reshaped daily_bikes tensor.\nRemember that it’s shaped (B, C, L), where L = 24. First, create the zero tensor, with\nthe same B and L but with the number of additional columns as C:\n'

In [18]:
daily_weather_onehot = torch.zeros(daily_bikes.shape[0], 4, daily_bikes.shape[2])
daily_weather_onehot.shape

torch.Size([730, 4, 24])

In [ ]:
"""
Then scatter the one-hot encoding into the tensor in the C dimension. Because operation
is performed in place, only the content of the tensor changes:
"""

In [19]:
daily_weather_onehot.scatter_(1, daily_bikes[:, 9, :].long().unsqueeze(1) - 1, 1.0)

tensor([[[1., 1., 1.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 1., 1., 1.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[0., 0., 0.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        ...,

        [[0., 0., 0.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[0., 0., 0.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0

In [20]:
daily_weather_onehot.shape

torch.Size([730, 4, 24])

In [21]:
daily_bikes = torch.cat((daily_bikes, daily_weather_onehot), dim=1)

In [22]:
daily_bikes.shape

torch.Size([730, 21, 24])

In [ ]:
"""
We mentioned earlier that this method isn’t the only way to treat the weather-situation
variable. Indeed, its labels have an ordinal relationship, so you could pretend that
they’re special values of a continuous variable. You might transform the variable so
that it runs from 0.0 to 1.0:
"""

In [24]:
daily_bikes[:, 9, :] = (daily_bikes[:, 9, :] - 1.0) / 3.0

In [ ]:
"""
As we mention in section 4.1, rescaling variables to the [0.0, 1.0] interval or the
[-1.0, 1.0] interval is something that you’ll want to do for all quantitative variables,
such as temperature (column 10 in your data set). You’ll see why later; for now, we’ll
say that it’s beneficial to the training process.
You have multiple possibilities for rescaling variables. You can map their range to
[0.0, 1.0]
"""

In [25]:
temp = daily_bikes[:, 10, :]
temp_min = torch.min(temp)
temp_max = torch.max(temp)
daily_bikes[:, 10, :] = (daily_bikes[:, 10, :] - temp_min) / (temp_max - temp_min)

In [ ]:
"""
or subtract the mean and divide by the standard deviation:
"""

In [29]:
temp = daily_bikes[:, 10, :]
daily_bikes[:, 10, :] = (daily_bikes[:, 10, :] - torch.mean(temp)) / torch.std(temp)

In [ ]:
"""
In this latter case, the variable has zero mean and unitary standard deviation. If the
variable were drawn from a Gaussian distribution, 68 percent of the samples would sit
in the [-1.0, 1.0] interval.
Great—you’ve built another nice data set that you’ll get to use later. For now, it’s
important only that you got an idea of how a time series is laid out and how you can
wrangle the data into a form that a network will digest.
Other kinds of data look like a time series, in that strict ordering exists. The top
two in that category are text and audio.
"""